# Phase 1 — train the semantic baseline on Kaggle

This machine has the GPUs; the code lives on GitHub. The notebook installs the
package, trains one fold, scores it, and writes a submission. It holds no
model code of its own, so there is never a second copy to keep in step.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on** (needed to install from GitHub)
3. Add the competition data as an input

Save it with **Save Version**, and under *Advanced Settings* set **Save
output** to keep it. The default on a Quick Save is to discard the output,
which throws away the checkpoint and the probability maps this notebook
exists to produce.

Expected wall clock on one T4: about 36 minutes for 15 epochs at 1024 pixels,
one minute to score the validation fold, four to write its probability maps and
four for the test set.

## 1. Clone the repository and put it on the path

The repository is cloned rather than installed from its URL, because
`configs/paths.yaml` and the frozen splits in `configs/splits/` sit beside the
package rather than inside it. A wheel would leave them behind, and the run
would then be validated on a different set of frames than every other run.

The clone is added to `sys.path` rather than installed with pip. This is a pure
Python package in a `src` layout, so the import works either way, and skipping
pip skips its checks on the interpreter version -- Kaggle's Python moves
independently of the one this is developed on. It also makes the repository
root resolve to the clone, which is how `configs/` is found.

Only the dependencies Kaggle does not already have are installed. Kaggle ships
a CUDA build of torch and replacing it costs minutes and risks a mismatch with
the driver, so torch is left alone.

To update: push, then rerun this cell. To reproduce an old run, put its commit
hash in `REF`.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "main"  # or a commit hash, for a run you will want to reproduce later
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q segmentation-models-pytorch

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import torch

import filament

print("filament", filament.__version__)
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("name:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

The package reads its paths from `configs/paths.yaml`, and the environment
variable `MAGFILO_ROOT` overrides the dataset root. Rather than hard-coding a
Kaggle input path, the annotation file is searched for: the input directory is
named after whatever the competition data was attached as.

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

# <root>/train/<annotations> -> <root>
dataset_root = candidates[0].parent.parent
os.environ["MAGFILO_ROOT"] = str(dataset_root)
print("MAGFILO_ROOT =", dataset_root)

# load_paths reads the environment variable when it is called, not at import.
paths = load_paths().require_dataset()
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))
print("test images: ", len(list(paths.test_images.glob("*.jpeg"))))

## 3. Train

The split is the frozen one committed in `configs/splits/`, so this fold is the
same set of frames every run. `num_workers` is raised from the committed value:
that default is 0 for Windows, where every worker would re-read the 48 MB
annotation file.

`output_dir` points into `/kaggle/working`, which is what Kaggle keeps as the
notebook's output.

In [ ]:
from dataclasses import replace

from filament.training.config import TrainConfig
from filament.training.loop import train

config = replace(
    TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase1_unet.yaml"),
    num_workers=2,
    output_dir=Path("/kaggle/working/phase1_unet"),
)
print(config)

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

result = train(config)
print("best epoch", result.best_epoch, "val loss", round(result.best_val_loss, 4))
print("checkpoint", result.checkpoint)

In [ ]:
import matplotlib.pyplot as plt

epochs = [item.epoch for item in result.history]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(
    epochs,
    [item.train_loss for item in result.history],
    label="train",
    linewidth=2,
    color="#2a78d6",
)
ax.plot(
    epochs,
    [item.val_loss for item in result.history],
    label="validation",
    linewidth=2,
    color="#eb6834",
)
ax.set_xlabel("epoch")
ax.set_ylabel("Dice + BCE loss")
ax.set_title("Training and validation loss")
ax.legend(frameon=False)
ax.grid(axis="y", linewidth=0.8, color="#e3e2dd")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"total training time: {sum(item.seconds for item in result.history) / 60:.1f} min")

## 4. Score the validation fold

Panoptic Quality with its breakdown, plus the counts of fused and split
filaments. Those two counts are the point of this phase: connected regions
cannot separate two filaments that touch, and PQ charges for that without
saying so. The exit target is **PQ 0.25**.

In [ ]:
from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.evaluation import evaluate
from filament.training.loop import load_checkpoint

model, stored = load_checkpoint(result.checkpoint)
dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(config.fold, f"{CHECKOUT}/configs/splits").val
print(f"fold {config.fold}: {len(val_stems)} validation frames")

evaluation, predictions = evaluate(
    model,
    dataset,
    paths.train_images,
    val_stems,
    size=config.image_size,
    device="cuda",
)
print(evaluation)

In [ ]:
import json

summary = evaluation.to_dict()
Path("/kaggle/working/eval_fold0.json").write_text(json.dumps(summary, indent=2))
summary

## 4b. Save the validation probability maps

Post-processing is where the remaining score is: the run above fragments 156
ground-truth filaments across more than one prediction, against 21 it fuses,
and no amount of further training changes that. Tuning the chain that turns a
probability map into instances needs the maps, not the model.

Writing them out once means that work never needs a GPU again. Float16 keeps a
fold's worth to about 300 MB, which fits in the notebook output and can be
attached as an input to a later notebook.

In [ ]:
import numpy as np

from filament.data.image import load_grayscale
from filament.evaluation import predict_probability

maps_dir = Path("/kaggle/working/prob_fold0")
maps_dir.mkdir(parents=True, exist_ok=True)

for position, stem in enumerate(val_stems, start=1):
    probability = predict_probability(
        model,
        load_grayscale(paths.train_images / f"{stem}.jpeg"),
        size=config.image_size,
        device="cuda",
    )
    np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    if position % 40 == 0:
        print(f"{position}/{len(val_stems)}")

written = sorted(maps_dir.glob("*.npy"))
total = sum(path.stat().st_size for path in written) / 1e6
print(f"{len(written)} maps, {total:.0f} MB")

## 5. Predict the test set

The overlap check is not optional: Kaggle rejects a submission whose masks
share a pixel, and the rejected attempt still uses one of the five allowed
per day.

In [ ]:
import pandas as pd

from filament.evaluation import predict_frame
from filament.metrics.overlap import check_no_overlap
from filament.postprocess.instances import instances_to_rows
from filament.submit.rle import write_submission

test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
rows = []
for position, stem in enumerate(test_stems, start=1):
    instances = predict_frame(
        model, paths.test_images / f"{stem}.jpeg", size=config.image_size, device="cuda"
    )
    rows.extend(instances_to_rows(stem, instances))
    if position % 30 == 0:
        print(f"{position}/{len(test_stems)}")

submission = Path("/kaggle/working/submission.csv")
write_submission(rows, submission)
check_no_overlap(submission)
print(f"{len(rows)} predictions over {len(test_stems)} frames")
pd.read_csv(submission).head()

## 6. What to record

Copy these into the lab notebook entry for this run, alongside the commit hash
the notebook installed:

- PQ, SQ, RQ, TP, FP, FN
- the fused and split counts
- mean IoU and mean Dice over the matches
- training time, and inference time for the validation fold
- the leaderboard score once the submission is in, next to the local PQ

The gap between the local score and the leaderboard is worth watching. Part of
the leaderboard is inflated by an overlap between the test set and a public
release of this dataset, which this project does not use, so some difference is
expected. A gap above 0.1 means something is wrong with the split or the
submission format instead.